# SAP — Carga silver (planilha de unidades prisionais)

Materializa a planilha de unidades prisionais da SAP-SP (município, código
IBGE, regional, RAJ, comarca e dados operacionais) na tabela `sap_unidade`
do Warehouse `mp_silver`, usando `python/src/modulos/sap/` do repositório
`mpsp-jurimetria/proj202607`.

**Pré-requisitos:**
- Subir a planilha `sap_nova_corrigida.xlsx` para o Lakehouse `mp_bronze`,
  seção Files, no caminho `sap/sap_nova_corrigida.xlsx` (upload manual pelo
  portal: mp_bronze > Files > pasta `sap` > Upload). Repetir o upload a cada
  nova versão da planilha e rodar este notebook de novo.
- Identidade do notebook com leitura no `mp_bronze` e escrita no `mp_silver`.

Depois desta carga, rodar `../cnmp/02_carga_gold.ipynb` (a dim_unidade é
enriquecida com as colunas da SAP via de-para curado) e, se o modelo
semântico precisar refletir colunas novas, `../cnmp/03_modelo_semantico.ipynb`.

In [ ]:
%pip install --quiet git+https://github.com/mpsp-jurimetria/proj202607.git#subdirectory=python

## Configuração

Mesma observação dos outros notebooks: não são segredos, mas evite deixar
valores reais commitados aqui.

In [ ]:
import os

os.environ["FABRIC_WORKSPACE_ID"] = "<id do workspace>"
os.environ["FABRIC_LAKEHOUSE_ID"] = "<id do lakehouse mp_bronze>"
os.environ["FABRIC_WAREHOUSE_SILVER_HOST"] = "<host>.datawarehouse.fabric.microsoft.com"
os.environ["FABRIC_WAREHOUSE_SILVER_NAME"] = "mp_silver"

In [ ]:
from src.infra.warehouse import get_silver_engine
from src.modulos.sap.etl.load_silver import carregar_silver

engine = get_silver_engine()
carregar_silver(engine)

## Verificação

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM sap_unidade")).scalar()
    print(f"sap_unidade: {total} unidades")
    amostra = conn.execute(text(
        "SELECT TOP 5 unidade_nome, municipio, regional, raj FROM sap_unidade"
    )).all()
    for linha in amostra:
        print(linha)